In [1]:
import os
import re

import numpy as np
import torch
from einops import rearrange

from dataloader import LifeGameDataset
from model_conv import *

s = r"D:\Internship\bimsa\result\predictor_life_simple\2026-02-10_17-25-43_small_4_layer_seq_cnn__200-200-B3678_S34678\best_simple_life_SimpleCNNSmall_4Layer_0.1.0.pth"
d = r"D:\Internship\bimsa\predictor_life_simple\datasets\200-200-B3678_S34678"

model = SimpleCNNSmall_4Layer()

all_files = [os.path.join(d, f) for f in os.listdir(d) if f.endswith('.npy')]
all_files.sort()  # Ensure deterministic split
test_files = all_files[::5]
file_list = [i for i in all_files if i not in test_files]

dataset = LifeGameDataset(file_list=file_list)
data = np.stack(dataset.arr_list, axis=1).reshape(-1, 1, 200, 200)

model.load_state_dict(torch.load(s))
input_original_tensors = torch.tensor(data).float().requires_grad_(True)

print(input_original_tensors.shape)
input_tensors = rearrange(torch.nn.Unfold(kernel_size=(11, 11), padding=0, stride=15)(input_original_tensors),
                          "n (a b) l -> (n l) 1 a b", a=11, b=11)
print(input_tensors.shape)

out = model(input_tensors)
*_, row, col = out.shape
row, col
grad = torch.zeros_like(out)
grad[0, 1, row//2, col//2] = 1

out.backward(gradient=grad)
print(input_tensors.grad.shape)
grad_abs = input_tensors.grad.abs().clone()

res_max, res_avg = torch.max(grad_abs, dim=0)[0][0].numpy(), torch.mean(grad_abs, dim=(0, 1)).numpy()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 8), dpi=300)
plt.suptitle(f"Everage and Maximum Effective Respective Fields of {model.__class__.__name__} trained on B3678/S34678 Data")

ax1 = plt.subplot(1, 2, 1)
ax1.set_title(r"Maximum Absolute ERF accross 3200 samples")
annot_matrix = np.where(np.abs(res_max) >= 0.01, np.round(res_max, 2), '')
sns.heatmap(res_max, cmap="Blues", annot=annot_matrix, fmt='', ax=ax1, vmax=5, linewidths=1, linecolor="gray")
ax1.set_xticks([]); ax1.set_yticks([])

ax2 = plt.subplot(1, 2, 2)
ax2.set_title(r"Average Absolute ERF accross 3200 samples")
annot_matrix = np.where(np.abs(res_avg) >= 0.01, np.round(res_avg, 2), '')
sns.heatmap(res_avg, cmap="Blues", annot=annot_matrix, fmt='', ax=ax2, vmax=5, linewidths=1, linecolor="gray")
ax2.set_xticks([]); ax2.set_yticks([])


torch.Size([3200, 1, 200, 200])
torch.Size([540800, 1, 11, 11])


C:\Users\Gravitas\AppData\Local\Temp\ipykernel_5772\785200493.py:39: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  print(input_tensors.grad.shape)


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
res_avg.max()